# Sample Band Comparison: JEFF/JENDL Bands vs. This Work MC Samples

3x2 comparison of Fe-56 elastic scattering angular distributions:
- **Top row**: JEFF-4.0 and JENDL-5 with MF34 uncertainty bands + EXFOR
- **Middle row**: Set (1) and Set (2) with MC sample curves + EXFOR
- **Bottom row**: Set (3) and Set (4) with MC sample curves + EXFOR

**Workflow**: Run cells 1-4 once (heavy loading), then change `TARGET_ENERGY_MEV` in cell 5 and re-run from there.

## 1. Configuration

In [ ]:
# ============================================================================
# USER CONFIGURABLE PARAMETERS
# ============================================================================

# -- ENDF evaluation files --------------------------------------------------
JEFF_FILE  = "/mnt/c/Users/MONLEON-DE-LA-JAN/Documents/endf_files/26-Fe-56g.txt"
JENDL_FILE = "/mnt/c/Users/MONLEON-DE-LA-JAN/Documents/endf_files/260560.jendl5"

# original set
SET1_NOMINAL = "/share_snc/snc/JuanMonleon/ENDF_samples/new_endf/nominal/26-Fe-56g_nominal.endf"
SET1_PARQUET = "/share_snc/snc/JuanMonleon/ENDF_samples/new_endf/nominal/legendre_coefficients_all_samples.parquet"
# without not fixed
SET2_NOMINAL = "/share_snc/snc/JuanMonleon/ENDF_samples/without/nominal/26-Fe-56g_nominal_mg.endf"
SET2_PARQUET = "/share_snc/snc/JuanMonleon/ENDF_samples/without/legendre_coefficients_all_samples.parquet"
# CAP3
SET3_NOMINAL = "/share_snc/snc/JuanMonleon/ENDF_samples/cap3/26-Fe-56g_nominal_mg.endf"
SET3_PARQUET = "/share_snc/snc/JuanMonleon/ENDF_samples/cap3/legendre_coefficients_all_samples.parquet"
# Without-fixed
SET4_NOMINAL = "/share_snc/snc/JuanMonleon/ENDF_samples/without-fixed/26-Fe-56g_nominal_mg.endf"
SET4_PARQUET = "/share_snc/snc/JuanMonleon/ENDF_samples/without-fixed/legendre_coefficients_all_samples.parquet"

# MT number for elastic scattering
MT_NUMBER = 2

# -- Active cases -----------------------------------------------------------
# Comment out lines below to disable individual cases.
EVAL_CASES = {
    'JEFF-4.0': JEFF_FILE,
    'JENDL-5':  JENDL_FILE,
}
SAMPLE_CASES = {
    'Set (1)': (SET1_NOMINAL, SET1_PARQUET),
    'Set (2)': (SET2_NOMINAL, SET2_PARQUET),
    'Set (3)': (SET3_NOMINAL, SET3_PARQUET),
    'Set (4)': (SET4_NOMINAL, SET4_PARQUET),
}

# -- EXFOR configuration ----------------------------------------------------
EXFOR_DB_PATH = '/share_snc/snc/JuanMonleon/EXFOR/x4_iron_angular.db'
TARGET_ZAIDS  = [26056, 26000]  # Fe-56 + natural iron

SUPPLEMENTARY_JSON_FILES = [
    '/share_snc/snc/JuanMonleon/EXFOR/data_v1/27673002.json',
]
EXCLUDE_EXPERIMENTS = ["32246002"]
MIN_RELATIVE_UNCERTAINTY = 0.03

# -- Normalization rules -----------------------------------------------------
KINNEY_SUBENTRY      = "10571002"
SMITH_SUBENTRY       = "10886002"
ENERGY_THRESHOLD_MEV = 2.5

# -- Physics parameters ------------------------------------------------------
M_PROJ_U = 1.008665   # neutron mass (u)
M_TARG_U = 55.93494   # Fe-56 mass (u)
DELTA_T_NS    = 5.0
FLIGHT_PATH_M = 27.037

# -- Plot settings -----------------------------------------------------------
N_SIGMA          = 1
PLOT_YSCALE      = 'log'
N_SAMPLES_PLOT   = 100      # number of sample curves to draw
SAMPLE_ALPHA     = 0.08     # transparency for each sample curve
EXFOR_PLOT_N_SIGMA   = 2.0
EXFOR_PLOT_MIN_ALPHA = 0.3

FIGURE_DPI = 300
FIGSIZE_3x2 = (10, 12)

FONTSIZES = {'axes_label': 10, 'tick_label': 9, 'legend': 8, 'subplot_title': 11}

CASE_COLORS = {
    'JEFF-4.0': 'tab:blue',
    'JENDL-5':  'tab:green',
    'Set (1)':  'tab:red',
    'Set (2)':  'tab:orange',
    'Set (3)':  'tab:purple',
    'Set (4)':  'tab:brown',
}

## 2. Imports

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from numpy.polynomial.legendre import legval
from scipy.special import legendre

# Ensure kika is on the path
_kika_path = Path().absolute().parent.parent.parent.parent
if str(_kika_path) not in sys.path:
    sys.path.insert(0, str(_kika_path))

from kika.endf import read_endf
from kika.exfor import read_all_exfor
import kika.exfor as exfor
from kika.plotting.styles import _get_color_palette

from scripts.exfor_utils import (
    build_exfor_cache_from_objects,
    filter_exfor_with_energy_bin,
    filter_exfor_with_kernel_weights,
)
from scripts.resample_AD import (
    compute_energy_resolution_tof as compute_sigma_E,
    sample_legendre_coefficients,
)

exfor.configure(db_path=EXFOR_DB_PATH)

plt.rcParams['figure.dpi'] = FIGURE_DPI
KIKA_COLORS = _get_color_palette('light')
EXFOR_MARKERS = ['o', 's', '^', 'v', 'D', 'p', 'h', '*', 'X', '<', '>']

print("Imports OK")

Imports OK


## 3. Load Heavy Data (run once)

In [ ]:
def load_endf_raw(filepath, label):
    "Load an ENDF file (expensive I/O). Run once."
    print(f"  Loading {label} from {filepath} ...")
    endf_obj = read_endf(filepath)
    mt_section = endf_obj.get_file(4).sections[MT_NUMBER]
    energies_ev = np.array(mt_section.legendre_energies)
    mf34 = None
    try:
        mf34 = endf_obj.mf[34].mt[MT_NUMBER].to_ang_covmat()
        print(f"    MF34: {mf34.num_matrices} matrices")
    except Exception as exc:
        print(f"    MF34: not available ({exc})")
    return dict(label=label, endf=endf_obj, mf34=mf34,
                energies_mev=energies_ev / 1e6, mt_section=mt_section)


# Load ENDF files
print("Loading ENDF evaluation files...")
endf_raw = {}
for key, filepath in EVAL_CASES.items():
    endf_raw[key] = load_endf_raw(filepath, key)
for key, (nominal, _parquet) in SAMPLE_CASES.items():
    endf_raw[key] = load_endf_raw(nominal, key)
print(f"Loaded {len(endf_raw)} ENDF files.\n")

# Load parquet files
print("Loading parquet sample files...")
parquet_data = {}
for key, (_nominal, parquet_path) in SAMPLE_CASES.items():
    parquet_data[key] = pd.read_parquet(parquet_path)
    df = parquet_data[key]
    n_samples = df['sample_idx'].max()
    n_energies = df['energy_index'].nunique()
    print(f"  {key}: {n_samples} MC samples, {n_energies} energies")

# Load EXFOR database
print("\nLoading EXFOR data from database...")
exfor_dict = read_all_exfor(
    target=TARGET_ZAIDS, mt=MT_NUMBER, source="database",
    group_by_energy=False,
    supplementary_json_files=SUPPLEMENTARY_JSON_FILES,
    exclude_experiments=EXCLUDE_EXPERIMENTS,
)
exfor_objects = list(exfor_dict.values())
exfor_cache, sorted_exfor_energies = build_exfor_cache_from_objects(
    exfor_objects, exclude_experiments=EXCLUDE_EXPERIMENTS,
)
print(f"  Datasets        : {len(exfor_objects)}")
print(f"  Unique energies : {len(sorted_exfor_energies)}")
print(f"  Energy range    : [{min(sorted_exfor_energies):.4f}, {max(sorted_exfor_energies):.4f}] MeV")
print("\nAll data loaded.")

## 4. Helper Functions

In [4]:
def select_energy_from_endf(raw_case, target_energy_mev):
    "Select target energy bin from pre-loaded ENDF (cheap). Re-run per energy."
    energies_mev = raw_case['energies_mev']
    target_idx = int(np.argmin(np.abs(energies_mev - target_energy_mev)))
    closest_mev = energies_mev[target_idx]
    endf_coeffs = np.array(raw_case['mt_section'].legendre_coefficients[target_idx])
    L_max = len(endf_coeffs)
    if target_idx > 0:
        bin_lower = (energies_mev[target_idx - 1] + closest_mev) / 2
    else:
        bin_lower = 0.0
    if target_idx < len(energies_mev) - 1:
        bin_upper = (closest_mev + energies_mev[target_idx + 1]) / 2
    else:
        bin_upper = float('inf')
    print(f"  {raw_case['label']}: E={closest_mev:.6f} MeV, L_max={L_max}, "
          f"bin=[{bin_lower:.6f}, {bin_upper:.6f}]")
    return dict(**raw_case, L_max=L_max, endf_coeffs=endf_coeffs,
                target_idx=target_idx, closest_energy_mev=closest_mev,
                bin_lower=bin_lower, bin_upper=bin_upper)


def filter_exfor_for_case(case_data, post_filter_subentry=None):
    "Filter EXFOR for a given case energy bin, optionally restricting to one subentry."
    df, exp_info, kw, diag = filter_exfor_with_energy_bin(
        exfor_cache=exfor_cache,
        sorted_energies=sorted_exfor_energies,
        bin_lower_mev=case_data['bin_lower'],
        bin_upper_mev=case_data['bin_upper'],
        target_energy_mev=case_data['closest_energy_mev'],
        m_proj_u=M_PROJ_U, m_targ_u=M_TARG_U,
        dedupe_per_experiment=True,
        exclude_experiments=EXCLUDE_EXPERIMENTS,
        min_relative_uncertainty=MIN_RELATIVE_UNCERTAINTY,
        normalize_by_n_points=True,
        max_experiment_weight_fraction=0.5,
    )
    if not df.empty and 'experiment_id' not in df.columns:
        df['experiment_id'] = df['entry'] + '/' + df['subentry']
    if post_filter_subentry is not None and not df.empty:
        entry_str = post_filter_subentry[:5]
        sub_str   = post_filter_subentry[5:]
        mask = (df['entry'] == entry_str) & (df['subentry'] == sub_str)
        if mask.any():
            df  = df[mask].reset_index(drop=True)
            kw  = kw[mask.values] if kw is not None else None
            exp_info = [e for e in exp_info
                        if e.get('entry') == entry_str and e.get('subentry') == sub_str]
        else:
            print(f"    WARNING: subentry {entry_str}/{sub_str} not found; using all data")
    return df, exp_info, kw


def compute_hybrid_baseline(case_data, exfor_df, kernel_weights):
    "Fit EXFOR to get c0, then combine with ENDF Legendre coefficients."
    degree = case_data['L_max']
    endf_coeffs = case_data['endf_coeffs']
    if exfor_df is None or exfor_df.empty or len(exfor_df) < 2:
        print(f"  WARNING: too few EXFOR points ({0 if exfor_df is None else len(exfor_df)})")
        return None
    coef_df, fit_info = sample_legendre_coefficients(
        df=exfor_df, value_col="value", unc_col="unc", mu_col="mu",
        degree=degree, n_samples=1,
        external_weights=kernel_weights,
        ridge_lambda=1e-6, rescale_unc_by_chi2=True,
    )
    c0 = coef_df['c0'].values[0]
    hybrid_coeffs = [c0]
    for l in range(1, degree + 1):
        a_l = endf_coeffs[l - 1]
        c_l = c0 * (2 * l + 1) * a_l
        hybrid_coeffs.append(c_l)
    print(f"  c0 (EXFOR) = {c0:.6f} b/sr, chi2_red = {fit_info['chi2_red']:.2f}")
    return dict(c0=c0, hybrid_coeffs=hybrid_coeffs, endf_coeffs=endf_coeffs,
                fit_info=fit_info, degree=degree)


def get_legendre_coefficient_uncertainty(mf34_covmat, endf_data, zaid, mt, energy_mev, order):
    "Get ABSOLUTE uncertainty for a single Legendre coefficient at a given energy."
    if mf34_covmat is None:
        return {'absolute': 0.0, 'relative': 0.0, 'is_relative': False}
    try:
        unc_data = mf34_covmat.get_uncertainties_for_legendre_coefficient(zaid, mt, order)
        if unc_data is None:
            return {'absolute': 0.0, 'relative': 0.0, 'is_relative': False}
        energies    = unc_data['energies']
        unc_values  = unc_data['uncertainties']
        is_relative = unc_data['is_relative']
        energy_ev   = energy_mev * 1e6
        bin_idx = -1
        for i in range(len(energies) - 1):
            if energies[i] <= energy_ev < energies[i + 1]:
                bin_idx = i
                break
        else:
            if abs(energy_ev - energies[-1]) < 1e-6:
                bin_idx = len(unc_values) - 1
        if bin_idx < 0:
            return {'absolute': 0.0, 'relative': 0.0, 'is_relative': False}
        unc_at_energy = unc_values[bin_idx]
        if not is_relative:
            return {'absolute': unc_at_energy, 'relative': 0.0, 'is_relative': False}
        mf4_mt = endf_data.mf[4].mt[mt]
        coeffs_dict = mf4_mt.extract_legendre_coefficients(
            energy_ev, max_legendre_order=order, out_of_range="zero")
        if coeffs_dict is not None and order in coeffs_dict:
            A_l = coeffs_dict[order]
            return {'absolute': unc_at_energy * abs(A_l),
                    'relative': unc_at_energy, 'is_relative': True}
        return {'absolute': 0.0, 'relative': unc_at_energy, 'is_relative': True}
    except Exception as exc:
        print(f"  Warning L={order}: {exc}")
        return {'absolute': 0.0, 'relative': 0.0, 'is_relative': False}


def extract_all_mf34_uncertainties(case_data):
    "Return {order: delta_a_abs} for l = 1 .. L_max."
    result = {}
    for l in range(1, case_data['L_max'] + 1):
        info = get_legendre_coefficient_uncertainty(
            case_data['mf34'], case_data['endf'], 26056, MT_NUMBER,
            case_data['closest_energy_mev'], l)
        result[l] = info['absolute']
    return result


def extract_mf34_covariance_matrix_at_energy(mf34_covmat, endf_data, zaid, mt, energy_mev, L_max):
    "Extract L_max x L_max absolute covariance matrix Cov[a_l, a_k] at one energy."
    cov = np.zeros((L_max, L_max))
    if mf34_covmat is None:
        return cov
    energy_ev = energy_mev * 1e6
    mf4_mt = endf_data.mf[4].mt[mt]
    coeffs_dict = mf4_mt.extract_legendre_coefficients(
        energy_ev, max_legendre_order=L_max, out_of_range="zero")
    for i in range(mf34_covmat.num_matrices):
        if (mf34_covmat.isotope_rows[i] != zaid or
            mf34_covmat.reaction_rows[i] != mt or
            mf34_covmat.isotope_cols[i] != zaid or
            mf34_covmat.reaction_cols[i] != mt):
            continue
        l = mf34_covmat.l_rows[i]
        k = mf34_covmat.l_cols[i]
        if l < 1 or l > L_max or k < 1 or k > L_max:
            continue
        egrid = np.asarray(mf34_covmat.energy_grids[i])
        bin_idx = -1
        for b in range(len(egrid) - 1):
            if egrid[b] <= energy_ev < egrid[b + 1]:
                bin_idx = b
                break
        if bin_idx < 0 and abs(energy_ev - egrid[-1]) < 1e-6:
            bin_idx = len(egrid) - 2
        if bin_idx < 0:
            continue
        value = float(mf34_covmat.matrices[i][bin_idx, bin_idx])
        if mf34_covmat.is_relative[i]:
            a_l = coeffs_dict.get(l, 0.0) if coeffs_dict else 0.0
            a_k = coeffs_dict.get(k, 0.0) if coeffs_dict else 0.0
            value *= abs(a_l) * abs(a_k)
        cov[l - 1, k - 1] = value
        cov[k - 1, l - 1] = value
    return cov


def extract_all_mf34_covariance(case_data):
    "Extract diagonal uncertainties dict AND full covariance matrix."
    delta_a_dict = extract_all_mf34_uncertainties(case_data)
    cov_matrix = extract_mf34_covariance_matrix_at_energy(
        case_data['mf34'], case_data['endf'], 26056, MT_NUMBER,
        case_data['closest_energy_mev'], case_data['L_max'])
    return delta_a_dict, cov_matrix


def compute_uncertainty_breakdown(fit_result, delta_a_dict, n_sigma, cov_matrix=None, n_points=200):
    "Compute baseline and uncertainty bands."
    c0             = fit_result['c0']
    hybrid_coeffs  = fit_result['hybrid_coeffs']
    L_max          = len(hybrid_coeffs) - 1
    mu             = np.linspace(-1, 1, n_points)
    baseline       = legval(mu, hybrid_coeffs)
    combined_diag  = np.zeros_like(mu)
    for l in range(1, L_max + 1):
        da = delta_a_dict.get(l, 0.0)
        combined_diag += (c0 * (2 * l + 1) * legendre(l)(mu) * da) ** 2
    combined_diag = np.sqrt(combined_diag)
    if cov_matrix is not None:
        S = np.column_stack([c0 * (2*l+1) * legendre(l)(mu) for l in range(1, L_max+1)])
        combined_full = np.sqrt(np.maximum(np.sum((S @ cov_matrix) * S, axis=1), 0.0))
    else:
        combined_full = combined_diag
    return dict(mu=mu, baseline=baseline, combined=combined_diag,
                combined_full=combined_full, n_sigma=n_sigma, L_max=L_max, c0=c0)


def compute_sample_curves(c0, parquet_df, target_energy_mev, mu, n_samples=100):
    "Compute dσ/dΩ curves for MC samples from parquet data at target energy."
    # Find the closest energy in the parquet
    energies = parquet_df[parquet_df['sample_idx'] == 0]['energy_mev'].values
    closest_idx = int(np.argmin(np.abs(energies - target_energy_mev)))
    closest_energy = energies[closest_idx]
    energy_index = parquet_df[parquet_df['sample_idx'] == 0].iloc[closest_idx]['energy_index']
    print(f"  Parquet closest energy: {closest_energy:.6f} MeV (index {int(energy_index)})")

    # Get a_l column names
    a_cols = [c for c in parquet_df.columns if c.startswith('a_')]
    a_cols = sorted(a_cols, key=lambda x: int(x.split('_')[1]))

    # Get sample rows at this energy
    mask = (parquet_df['energy_index'] == energy_index) & (parquet_df['sample_idx'] > 0)
    sample_rows = parquet_df.loc[mask]
    available = len(sample_rows)
    n_draw = min(n_samples, available)
    if n_draw < n_samples:
        print(f"    Only {available} samples available (requested {n_samples})")
    sample_rows = sample_rows.head(n_draw)

    # Build curves: dσ/dΩ = legval(mu, [c0, c0*3*a1, c0*5*a2, ...])
    curves = np.empty((n_draw, len(mu)))
    for i, (_, row) in enumerate(sample_rows.iterrows()):
        a_vals = row[a_cols].values.astype(float)
        # Strip trailing zeros
        last_nonzero = len(a_vals)
        while last_nonzero > 0 and a_vals[last_nonzero - 1] == 0:
            last_nonzero -= 1
        a_vals = a_vals[:max(last_nonzero, 1)]
        hybrid = [c0]
        for l_idx, a_l in enumerate(a_vals):
            l = l_idx + 1
            hybrid.append(c0 * (2 * l + 1) * a_l)
        curves[i] = legval(mu, hybrid)

    # Also get nominal (sample_idx=0)
    nom_mask = (parquet_df['energy_index'] == energy_index) & (parquet_df['sample_idx'] == 0)
    nom_row = parquet_df.loc[nom_mask]
    nom_a = nom_row[a_cols].values[0].astype(float)
    nom_hybrid = [c0]
    for l_idx, a_l in enumerate(nom_a):
        l = l_idx + 1
        if a_l == 0 and l_idx > 0 and all(nom_a[l_idx:] == 0):
            break
        nom_hybrid.append(c0 * (2 * l + 1) * a_l)
    nominal_curve = legval(mu, nom_hybrid)

    return curves, nominal_curve, closest_energy


def plot_exfor_overlay(ax, exfor_df, exp_info, ms=4, alpha=0.8, capsize=2, mew=0.5):
    "Overlay EXFOR data coloured by experiment."
    if exfor_df is None or exfor_df.empty:
        return
    for i, eid in enumerate(exfor_df['experiment_id'].unique()):
        m = exfor_df['experiment_id'] == eid
        info = next((e for e in exp_info
                     if f"{e.get('entry','')}/{e.get('subentry','')}" == eid), {})
        ax.errorbar(exfor_df.loc[m, 'mu'], exfor_df.loc[m, 'value'],
                    yerr=exfor_df.loc[m, 'unc'],
                    fmt=EXFOR_MARKERS[i % len(EXFOR_MARKERS)],
                    color=KIKA_COLORS[i % len(KIKA_COLORS)],
                    label=f"{info.get('author', eid)} ({info.get('year','?')})",
                    capsize=capsize, markersize=ms, alpha=alpha, markeredgewidth=mew)


print("Helper functions defined.")

Helper functions defined.


## 5. Set Target Energy -- Re-run from here

In [5]:
TARGET_ENERGY_MEV = 1.25

## 6. Energy-Dependent Processing

In [ ]:
# --- Select energy from ENDF ---
print("Selecting energy from ENDF evaluations...")
case_data = {}
for key in endf_raw:
    case_data[key] = select_energy_from_endf(endf_raw[key], TARGET_ENERGY_MEV)

# --- Common energy bin (union of all libraries) ---
common_bin_lower = min(d['bin_lower'] for d in case_data.values())
common_bin_upper = max(d['bin_upper'] for d in case_data.values())
# Use first sample case as reference, or first eval case
ref_key = next(iter(SAMPLE_CASES), next(iter(EVAL_CASES)))
common_target_energy = case_data[ref_key]['closest_energy_mev']
common_data = dict(bin_lower=common_bin_lower, bin_upper=common_bin_upper,
                   closest_energy_mev=common_target_energy)
print(f"  Common bin: [{common_bin_lower:.6f}, {common_bin_upper:.6f}] MeV")

# --- Filter EXFOR ---
sub = KINNEY_SUBENTRY if TARGET_ENERGY_MEV < ENERGY_THRESHOLD_MEV else SMITH_SUBENTRY
print(f"\nNormalization subentry: {sub}")

# Eval cases: use normalization subentry
print("\nFiltering EXFOR ...")
eval_exfor = {}
for key in EVAL_CASES:
    df, exp_info, kw = filter_exfor_for_case(case_data[key], sub)
    eval_exfor[key] = (df, exp_info, kw)
    print(f"  {key}: {len(df)} pts")

# Sample cases: all in-bin EXFOR (no subentry filter)
shared_exfor_all, shared_exp_info_all, shared_kw = filter_exfor_for_case(common_data, None)
print(f"  All in-bin EXFOR: {len(shared_exfor_all)} pts")

# --- Extended EXFOR for plotting (nearby points) ---
print("\nBuilding extended EXFOR ...")
sigma_E_mev = compute_sigma_E(
    common_target_energy, delta_t_ns=DELTA_T_NS, flight_path_m=FLIGHT_PATH_M)
ext_df, ext_exp_info, ext_kw, _ = filter_exfor_with_kernel_weights(
    exfor_cache=exfor_cache, sorted_energies=sorted_exfor_energies,
    energy_mev=common_target_energy, sigma_E_mev=sigma_E_mev,
    n_sigma=EXFOR_PLOT_N_SIGMA, m_proj_u=M_PROJ_U, m_targ_u=M_TARG_U,
    bin_lower_mev=common_bin_lower, bin_upper_mev=common_bin_upper,
    use_overlap_weights=False, dedupe_per_experiment=False,
    normalize_by_n_points=False, exclude_experiments=EXCLUDE_EXPERIMENTS,
    min_relative_uncertainty=MIN_RELATIVE_UNCERTAINTY, max_experiment_weight_fraction=1.0)

ext_exfor_data = None
if not ext_df.empty:
    if 'experiment_id' not in ext_df.columns:
        ext_df['experiment_id'] = ext_df['entry'] + '/' + ext_df['subentry']
    ext_df['in_bin'] = (
        (ext_df['exfor_energy_mev'] >= common_bin_lower) &
        (ext_df['exfor_energy_mev'] <= common_bin_upper))
    target_e = common_target_energy
    for eid in ext_df.loc[ext_df['in_bin'], 'experiment_id'].unique():
        exp_in_bin = (ext_df['experiment_id'] == eid) & ext_df['in_bin']
        unique_energies = ext_df.loc[exp_in_bin, 'exfor_energy_mev'].unique()
        if len(unique_energies) > 1:
            closest_energy = unique_energies[np.argmin(np.abs(unique_energies - target_e))]
            demote = exp_in_bin & (ext_df['exfor_energy_mev'] != closest_energy)
            ext_df.loc[demote, 'in_bin'] = False
    max_kw = ext_kw.max() if len(ext_kw) > 0 else 1.0
    norm_kw = ext_kw / max_kw if max_kw > 0 else np.ones_like(ext_kw)
    ext_df['plot_alpha'] = EXFOR_PLOT_MIN_ALPHA + norm_kw * (0.9 - EXFOR_PLOT_MIN_ALPHA)
    ext_df.loc[ext_df['in_bin'], 'plot_alpha'] = 0.9
    all_exp_ids = ext_df['experiment_id'].unique()
    in_bin_exp_ids = ext_df.loc[ext_df['in_bin'], 'experiment_id'].unique().tolist()
    nearby_only_exp_ids = [e for e in all_exp_ids if e not in in_bin_exp_ids]
    ordered_exp_ids = in_bin_exp_ids + nearby_only_exp_ids
    exp_visual_map = {}
    for i, eid in enumerate(ordered_exp_ids):
        exp_visual_map[eid] = (KIKA_COLORS[i % len(KIKA_COLORS)],
                               EXFOR_MARKERS[i % len(EXFOR_MARKERS)])
    exp_label_map = {}
    for info in ext_exp_info:
        eid = f"{info.get('entry', '')}/{info.get('subentry', '')}"
        exp_label_map[eid] = f"{info.get('author', eid)} ({info.get('year', '?')})"
    for eid in ordered_exp_ids:
        if eid not in exp_label_map:
            exp_label_map[eid] = eid
    ext_exfor_data = dict(
        ext_df=ext_df, exp_visual_map=exp_visual_map,
        exp_label_map=exp_label_map, in_bin_set=set(in_bin_exp_ids))
    n_nearby = (~ext_df['in_bin']).sum()
    n_inbin = ext_df['in_bin'].sum()
    print(f"  Extended: {n_inbin} in-bin + {n_nearby} nearby")

# --- Hybrid baselines ---
print("\nComputing hybrid baselines ...")
fit_results = {}
for key in EVAL_CASES:
    df, _exp_info, kw = eval_exfor[key]
    fit_results[key] = compute_hybrid_baseline(case_data[key], df, kw)
for key in SAMPLE_CASES:
    fit_results[key] = compute_hybrid_baseline(case_data[key], shared_exfor_all, shared_kw)

# --- MF34 uncertainties for eval cases ---
print("\nExtracting MF34 uncertainties ...")
eval_delta_a = {}
eval_cov = {}
for key in EVAL_CASES:
    eval_delta_a[key], eval_cov[key] = extract_all_mf34_covariance(case_data[key])

# --- Uncertainty breakdown for eval cases ---
print("\nComputing uncertainty breakdown ...")
eval_results = {}
for key in EVAL_CASES:
    eval_results[key] = compute_uncertainty_breakdown(
        fit_results[key], eval_delta_a[key], N_SIGMA, cov_matrix=eval_cov[key])

# --- Sample curves for sample cases ---
print("\nComputing sample curves ...")
mu = np.linspace(-1, 1, 200)
if eval_results:
    mu = next(iter(eval_results.values()))['mu']
sample_curves = {}
sample_nominals = {}
for key in SAMPLE_CASES:
    curves, nominal, parq_energy = compute_sample_curves(
        fit_results[key]['c0'], parquet_data[key], TARGET_ENERGY_MEV, mu, N_SAMPLES_PLOT)
    sample_curves[key] = curves
    sample_nominals[key] = nominal

print("\nProcessing complete.")

## 7. Plot 3x2 Grid

In [ ]:
# ---- Helper: plot EXFOR with extended nearby data on an axis ----
def _plot_ext_exfor(ax, ext_data):
    "Plot extended EXFOR (nearby in grey + in-bin in color) on axis."
    if ext_data is None:
        return
    edf = ext_data['ext_df']
    vmap = ext_data['exp_visual_map']
    lmap = ext_data['exp_label_map']
    _nearby_placed = False
    df_near = edf[~edf['in_bin']]
    for eid in df_near['experiment_id'].unique():
        mask = df_near['experiment_id'] == eid
        df_exp = df_near[mask]
        _, marker = vmap[eid]
        avg_alpha = df_exp['plot_alpha'].median()
        lbl = (f'Nearby EXFOR ({EXFOR_PLOT_N_SIGMA:.0f}$\\sigma_E$ window)'
               if not _nearby_placed else None)
        _nearby_placed = True
        ax.errorbar(df_exp['mu'], df_exp['value'], yerr=df_exp['unc'],
                    fmt=marker, color='0.55', label=lbl,
                    capsize=2, markersize=3, alpha=avg_alpha,
                    markerfacecolor='none', markeredgewidth=0.6, markeredgecolor='0.55',
                    elinewidth=0.5, ecolor='0.7')
    df_in = edf[edf['in_bin']]
    for eid in df_in['experiment_id'].unique():
        mask = df_in['experiment_id'] == eid
        df_exp = df_in[mask]
        color, marker = vmap[eid]
        ax.errorbar(df_exp['mu'], df_exp['value'], yerr=df_exp['unc'],
                    fmt=marker, color=color, label=lmap.get(eid, eid),
                    capsize=2, markersize=4, alpha=0.9, markeredgewidth=0.5)


def _plot_sample_panel(ax, key, mu_grid):
    "Plot MC sample curves + nominal on an axis."
    curves = sample_curves[key]
    nominal = sample_nominals[key]
    color = CASE_COLORS.get(key, 'tab:gray')
    for i in range(len(curves)):
        ax.plot(mu_grid, curves[i], color=color, alpha=SAMPLE_ALPHA, lw=0.5,
                label='MC samples' if i == 0 else None)
    ax.plot(mu_grid, nominal, 'k-', lw=1.5, label='Nominal')


def _plot_eval_panel(ax, key, mu_grid):
    "Plot MF34 uncertainty band + baseline on an axis."
    r = eval_results[key]
    band_full = r.get('combined_full', r['combined'])
    ns = r['n_sigma']
    color = CASE_COLORS.get(key, 'tab:gray')
    ax.fill_between(r['mu'], r['baseline'] - ns * band_full,
                    r['baseline'] + ns * band_full,
                    alpha=0.35, color=color,
                    label=f'{key} $\\pm{ns}\\sigma$')
    ax.plot(r['mu'], r['baseline'], 'k-', lw=1.5, label='Baseline')


def _make_dynamic_plot(yscale='log'):
    "Generate grid with one row per pair of cases."
    # Build ordered list of panels: evals first (paired), then samples (paired)
    eval_keys = list(EVAL_CASES.keys())
    sample_keys = list(SAMPLE_CASES.keys())
    # Group into rows of 2
    all_panels = eval_keys + sample_keys
    n_rows = (len(all_panels) + 1) // 2
    n_cols = min(2, len(all_panels))

    fig_height = 4 * n_rows
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, fig_height), sharex=True,
                             squeeze=False)
    fs = FONTSIZES

    # Collect all y-values for consistent axis limits
    all_y_list = []

    for panel_idx, key in enumerate(all_panels):
        row, col = divmod(panel_idx, 2)
        ax = axes[row, col]

        if key in eval_results:
            _plot_eval_panel(ax, key, mu)
            all_y_list.append(eval_results[key]['baseline'])
        elif key in sample_curves:
            _plot_sample_panel(ax, key, mu)
            all_y_list.append(sample_curves[key].ravel())

        _plot_ext_exfor(ax, ext_exfor_data)
        ax.set_title(key, fontsize=fs['subplot_title'])

    # Hide unused axes (odd number of panels)
    for panel_idx in range(len(all_panels), n_rows * n_cols):
        row, col = divmod(panel_idx, 2)
        axes[row, col].set_visible(False)

    # ---- Shared formatting ----
    all_y = np.concatenate(all_y_list) if all_y_list else np.array([1.0])
    all_y_pos = all_y[all_y > 0]

    for i in range(n_rows):
        for j in range(n_cols):
            ax = axes[i, j]
            if not ax.get_visible():
                continue
            ax.grid(True, alpha=0.3)
            ax.tick_params(labelsize=fs['tick_label'])
            if yscale == 'log':
                ax.set_yscale('log')
                if len(all_y_pos) > 0:
                    ax.set_ylim(all_y_pos.min() * 0.5, all_y_pos.max() * 2.0)
            else:
                if len(all_y_pos) > 0:
                    ax.set_ylim(all_y.min() * 0.9, all_y.max() * 1.1)
            if i == n_rows - 1:
                ax.set_xlabel(r'$\cos(\theta_{\rm CM})$', fontsize=fs['axes_label'])
            if j == 0:
                ax.set_ylabel(r'$d\sigma/d\Omega$ (b/sr)', fontsize=fs['axes_label'])

    # Shared legend
    handles, labels = [], []
    seen = set()
    for ax in axes.ravel():
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in seen:
                handles.append(h)
                labels.append(l)
                seen.add(l)

    fig.suptitle(f'Fe-56 Elastic Scattering at E = {TARGET_ENERGY_MEV:.2f} MeV ({yscale})',
                 fontsize=fs['subplot_title'] + 2, y=0.98)
    fig.tight_layout(rect=[0, 0.06, 1, 0.96])
    fig.legend(handles, labels, loc='upper center',
               bbox_to_anchor=(0.5, 0.05), ncol=min(4, len(labels)),
               fontsize=fs['legend'], frameon=True, fancybox=True)
    return fig


for yscale in ['log', 'linear']:
    fig = _make_dynamic_plot(yscale=yscale)
    plt.show()

In [ ]:
# ---- Uncertainty Summary for all cases ----
def _get_parquet_sample_stats(parquet_df, target_energy_mev):
    """Compute mean and std of a_l across MC samples at the closest energy."""
    energies = parquet_df[parquet_df['sample_idx'] == 0]['energy_mev'].values
    closest_idx = int(np.argmin(np.abs(energies - target_energy_mev)))
    energy_index = parquet_df[parquet_df['sample_idx'] == 0].iloc[closest_idx]['energy_index']
    a_cols = sorted([c for c in parquet_df.columns if c.startswith('a_')],
                    key=lambda x: int(x.split('_')[1]))
    # Nominal (sample_idx == 0)
    nom = parquet_df.loc[(parquet_df['energy_index'] == energy_index) &
                         (parquet_df['sample_idx'] == 0), a_cols].values[0].astype(float)
    # All samples (sample_idx > 0)
    samples = parquet_df.loc[(parquet_df['energy_index'] == energy_index) &
                             (parquet_df['sample_idx'] > 0), a_cols].values.astype(float)
    mean_a = samples.mean(axis=0)
    std_a  = samples.std(axis=0, ddof=1)
    n_samples = len(samples)
    # Find effective L_max (last non-zero std)
    last_nz = len(std_a)
    while last_nz > 0 and std_a[last_nz - 1] == 0 and nom[last_nz - 1] == 0:
        last_nz -= 1
    return nom[:last_nz], mean_a[:last_nz], std_a[:last_nz], n_samples

print("=" * 72)
print(f"  Legendre Coefficient Uncertainties at E = {TARGET_ENERGY_MEV:.2f} MeV")
print("=" * 72)

# ---- Eval cases (from MF34) ----
for lib_name in EVAL_CASES:
    delta_a = eval_delta_a[lib_name]
    cov = eval_cov[lib_name]
    cd = case_data[lib_name]
    endf_coeffs = cd['endf_coeffs']
    L_max = cd['L_max']
    print(f"\n{'─' * 72}")
    print(f"  {lib_name}  (L_max = {L_max}, E = {cd['closest_energy_mev']:.6f} MeV)")
    print(f"  Source: MF34 covariance")
    print(f"{'─' * 72}")
    print(f"  {'l':>3s}  {'a_l':>12s}  {'σ(a_l) abs':>12s}  {'σ(a_l) rel (%)':>14s}")
    print(f"  {'---':>3s}  {'------------':>12s}  {'------------':>12s}  {'--------------':>14s}")
    for l in range(1, L_max + 1):
        a_l = endf_coeffs[l - 1] if l - 1 < len(endf_coeffs) else 0.0
        da  = delta_a.get(l, 0.0)
        rel = (da / abs(a_l) * 100) if abs(a_l) > 1e-30 else float('nan')
        print(f"  {l:3d}  {a_l:12.6f}  {da:12.6e}  {rel:14.2f}")

    diag = np.sqrt(np.diag(cov[:L_max, :L_max]))
    if np.all(diag > 0):
        corr = cov[:L_max, :L_max] / np.outer(diag, diag)
        print(f"\n  Correlation matrix (l = 1..{L_max}):")
        header = "       " + "".join(f"  l={l:<5d}" for l in range(1, L_max + 1))
        print(header)
        for i in range(L_max):
            row_str = f"  l={i+1:<3d}"
            for j in range(L_max):
                row_str += f"  {corr[i, j]:+6.3f} "
            print(row_str)
    else:
        non_zero = [l+1 for l in range(L_max) if diag[l] > 0]
        print(f"\n  Non-zero uncertainties only for l = {non_zero}")

# ---- Sample cases (from MC samples) ----
for set_name in SAMPLE_CASES:
    pq_df = parquet_data[set_name]
    nom_a, mean_a, std_a, n_samp = _get_parquet_sample_stats(pq_df, TARGET_ENERGY_MEV)
    L_max_mc = len(nom_a)
    print(f"\n{'─' * 72}")
    print(f"  {set_name}  (L_max = {L_max_mc}, E = {TARGET_ENERGY_MEV:.6f} MeV)")
    print(f"  Source: MC samples (N = {n_samp}), 1σ = sample std")
    print(f"{'─' * 72}")
    print(f"  {'l':>3s}  {'a_l (nom)':>12s}  {'<a_l> (MC)':>12s}  {'σ(a_l) abs':>12s}  {'σ(a_l) rel (%)':>14s}")
    print(f"  {'---':>3s}  {'------------':>12s}  {'------------':>12s}  {'------------':>12s}  {'--------------':>14s}")
    for l_idx in range(L_max_mc):
        l = l_idx + 1
        a_nom = nom_a[l_idx]
        a_mean = mean_a[l_idx]
        da = std_a[l_idx]
        rel = (da / abs(a_nom) * 100) if abs(a_nom) > 1e-30 else float('nan')
        print(f"  {l:3d}  {a_nom:12.6f}  {a_mean:12.6f}  {da:12.6e}  {rel:14.2f}")

    # Correlation matrix from MC samples
    energies = pq_df[pq_df['sample_idx'] == 0]['energy_mev'].values
    ci = int(np.argmin(np.abs(energies - TARGET_ENERGY_MEV)))
    ei = pq_df[pq_df['sample_idx'] == 0].iloc[ci]['energy_index']
    mask = (pq_df['energy_index'] == ei) & (pq_df['sample_idx'] > 0)
    a_cols = sorted([c for c in pq_df.columns if c.startswith('a_')],
                    key=lambda x: int(x.split('_')[1]))
    samp_mat = pq_df.loc[mask, a_cols].values.astype(float)[:, :L_max_mc]
    if samp_mat.shape[0] > 2:
        mc_cov = np.cov(samp_mat, rowvar=False)
        mc_diag = np.sqrt(np.diag(mc_cov))
        nz = mc_diag > 0
        if nz.sum() >= 2:
            corr_mc = np.zeros_like(mc_cov)
            outer = np.outer(mc_diag, mc_diag)
            valid = outer > 0
            corr_mc[valid] = mc_cov[valid] / outer[valid]
            print(f"\n  Correlation matrix (l = 1..{L_max_mc}):")
            header = "       " + "".join(f"  l={l:<5d}" for l in range(1, L_max_mc + 1))
            print(header)
            for i in range(L_max_mc):
                row_str = f"  l={i+1:<3d}"
                for j in range(L_max_mc):
                    row_str += f"  {corr_mc[i, j]:+6.3f} "
                print(row_str)

print(f"\n{'=' * 72}")
print(f"  Note: JEFF/JENDL σ = sqrt(Cov[a_l,a_l]) from MF34.")
print(f"  Set (1)-(4) σ = std over MC samples from parquet.")
print(f"  The ±{N_SIGMA}σ bands (top row) use the FULL covariance (off-diag included).")
print(f"{'=' * 72}")